# 🔍 Vector Search Index Setup for Healthcare RAG

## Purpose
This notebook creates a Databricks Vector Search index on the `gold_facility_cards` table to enable semantic search over Ghana healthcare facilities.

## Architecture
**Input:** `vf_health.ghana.gold_facility_cards` (Gold layer table with `full_text_for_rag` column)  
**Embedding Model:** `databricks-gte-large-en` (Free tier compatible)  
**Index Type:** Delta Sync Index (automatically syncs with source table)  
**Primary Use:** LangChain agent tool for semantic facility search

## Outputs
- Vector Search Endpoint: `vf_health_vs_endpoint`
- Vector Search Index: `vf_health.ghana.facility_rag_index`
- Test query results demonstrating semantic search capabilities

In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
%python
# ============================================================================
# DATABRICKS VECTOR SEARCH SETUP
# ============================================================================
# Create a vector search index for semantic search over healthcare facilities

import time
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"
GOLD_FACILITY_CARDS = f"{CATALOG}.{SCHEMA}.gold_facility_cards"

VS_ENDPOINT_NAME = "vf_health_vs_endpoint"
VS_INDEX_NAME = f"{CATALOG}.{SCHEMA}.facility_rag_index"
EMBEDDING_SOURCE_COLUMN = "full_text_for_rag"
PRIMARY_KEY = "row_id"

print("="*70)
print("🔍 DATABRICKS VECTOR SEARCH INDEX CREATION")
print("="*70)

# Initialize Vector Search client
print("\n1️⃣ Initializing Vector Search client...")
vs_client = VectorSearchClient(disable_notice=True)

# ============================================================================
# STEP 1: CREATE OR GET VECTOR SEARCH ENDPOINT
# ============================================================================
print(f"\n2️⃣ Creating/Getting Vector Search endpoint: {VS_ENDPOINT_NAME}...")

try:
    endpoint = vs_client.get_endpoint(VS_ENDPOINT_NAME)
    print(f"   ✅ Endpoint '{VS_ENDPOINT_NAME}' already exists")
except Exception as e:
    if "does not exist" in str(e).lower() or "not found" in str(e).lower():
        print(f"   ⏳ Creating new endpoint '{VS_ENDPOINT_NAME}'...")
        try:
            endpoint = vs_client.create_endpoint(
                name=VS_ENDPOINT_NAME,
                endpoint_type="STANDARD"
            )
            # Wait for endpoint to be ready
            print("   ⏳ Waiting for endpoint to be ready...")
            for i in range(60):  # Wait up to 5 minutes
                endpoint_status = vs_client.get_endpoint(VS_ENDPOINT_NAME)
                if endpoint_status.get("endpoint_status", {}).get("state") == "ONLINE":
                    print(f"   ✅ Endpoint is ONLINE")
                    break
                print(f"      Waiting... ({i+1}/60)")
                time.sleep(5)
        except Exception as create_err:
            print(f"   ⚠️ Could not create endpoint: {create_err}")
            print("   ℹ️ On Databricks Free Edition, you may need to create the endpoint via UI")
            raise
    else:
        raise

# ============================================================================
# STEP 2: VERIFY SOURCE TABLE
# ============================================================================
print(f"\n3️⃣ Verifying source table: {GOLD_FACILITY_CARDS}...")

try:
    source_table = spark.table(GOLD_FACILITY_CARDS)
    row_count = source_table.count()
    print(f"   ✅ Source table exists with {row_count} rows")
    
    # Verify required columns
    required_cols = [PRIMARY_KEY, EMBEDDING_SOURCE_COLUMN]
    table_cols = source_table.columns
    
    for col in required_cols:
        if col not in table_cols:
            raise ValueError(f"Required column '{col}' not found in source table")
    
    print(f"   ✅ Required columns present: {', '.join(required_cols)}")
    
    # Show sample text lengths
    print("\n   📊 Sample text statistics:")
    source_table.select(
        F.min(F.length(EMBEDDING_SOURCE_COLUMN)).alias("min_len"),
        F.avg(F.length(EMBEDDING_SOURCE_COLUMN)).cast("int").alias("avg_len"),
        F.max(F.length(EMBEDDING_SOURCE_COLUMN)).alias("max_len")
    ).show()
    
except Exception as e:
    print(f"   ❌ Error: {e}")
    print("   ℹ️ Make sure to run 03_build_gold notebook first")
    raise

# ============================================================================
# STEP 3: CREATE OR UPDATE VECTOR SEARCH INDEX
# ============================================================================
print(f"\n4️⃣ Creating/Updating Vector Search index: {VS_INDEX_NAME}...")

try:
    # Try to get existing index
    existing_index = vs_client.get_index(VS_INDEX_NAME)
    print(f"   🔄 Index '{VS_INDEX_NAME}' already exists, will sync...")
    index_exists = True
except Exception:
    print(f"   🆕 Creating new index '{VS_INDEX_NAME}'...")
    index_exists = False

if not index_exists:
    try:
        index = vs_client.create_delta_sync_index(
            endpoint_name=VS_ENDPOINT_NAME,
            source_table_name=GOLD_FACILITY_CARDS,
            index_name=VS_INDEX_NAME,
            pipeline_type="TRIGGERED",  # Manual sync (not continuous)
            primary_key=PRIMARY_KEY,
            embedding_source_column=EMBEDDING_SOURCE_COLUMN,
            embedding_model_endpoint_name="databricks-gte-large-en"  # Free tier embedding model
        )
        print(f"   ✅ Index created successfully")
    except Exception as e:
        print(f"   ⚠️ Error creating index: {e}")
        print("\n   ℹ️ Troubleshooting tips:")
        print("      - Ensure endpoint is ONLINE")
        print("      - Verify table has correct permissions")
        print("      - Check that embedding model 'databricks-gte-large-en' is available")
        raise

# ============================================================================
# STEP 4: TRIGGER SYNC
# ============================================================================
print(f"\n5️⃣ Triggering index sync...")

try:
    vs_client.get_index(VS_INDEX_NAME).sync()
    print("   ⏳ Sync triggered, waiting for completion...")
    
    # Wait for sync to complete
    for i in range(60):  # Wait up to 5 minutes
        index_status = vs_client.get_index(VS_INDEX_NAME).describe()
        status_message = index_status.get("status", {}).get("message", "")
        
        if "ready" in status_message.lower() or "online" in status_message.lower():
            print(f"   ✅ Index sync complete! Status: {status_message}")
            break
        
        print(f"      Syncing... ({i+1}/60) - {status_message}")
        time.sleep(5)
    
    # Final status
    final_status = vs_client.get_index(VS_INDEX_NAME).describe()
    print("\n   📊 Final Index Status:")
    print(f"      Index Name: {final_status.get('name')}")
    print(f"      Status: {final_status.get('status', {}).get('message')}")
    
except Exception as e:
    print(f"   ⚠️ Sync may take time in background: {e}")
    print("   ℹ️ You can check status in the Vector Search UI")

print("\n="*70)
print("✓ VECTOR SEARCH INDEX SETUP COMPLETE")
print(f"  Endpoint: {VS_ENDPOINT_NAME}")
print(f"  Index: {VS_INDEX_NAME}")
print(f"  Source: {GOLD_FACILITY_CARDS}")
print(f"  Embedding Model: databricks-gte-large-en")
print("="*70)

In [0]:
# ============================================================================
# ENABLE CHANGE DATA FEED ON SOURCE TABLE
# ============================================================================
# Vector Search requires Change Data Feed to track table updates

CATALOG = "vf_health"
SCHEMA = "ghana"
GOLD_FACILITY_CARDS = f"{CATALOG}.{SCHEMA}.gold_facility_cards"

print("="*70)
print("📋 ENABLING CHANGE DATA FEED ON SOURCE TABLE")
print("="*70)

print(f"\n🔄 Enabling Change Data Feed on: {GOLD_FACILITY_CARDS}...")

try:
    spark.sql(f"""
        ALTER TABLE {GOLD_FACILITY_CARDS}
        SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    print(f"   ✅ Change Data Feed enabled successfully")
    
    # Verify the property
    table_props = spark.sql(f"SHOW TBLPROPERTIES {GOLD_FACILITY_CARDS}").collect()
    cdf_enabled = any(row[0] == 'delta.enableChangeDataFeed' and row[1] == 'true' for row in table_props)
    
    if cdf_enabled:
        print("   ✅ Verified: CDF property is set to 'true'")
    else:
        print("   ⚠️ Warning: Could not verify CDF property")
        
except Exception as e:
    print(f"   ❌ Error: {e}")
    raise

print("\n="*70)
print("✓ SOURCE TABLE READY FOR VECTOR SEARCH")
print("="*70)

In [0]:
# ============================================================================
# TEST VECTOR SEARCH INDEX
# ============================================================================
# Perform test queries to verify semantic search is working

from databricks.vector_search.client import VectorSearchClient
import pandas as pd
import time

# Define index and endpoint names
CATALOG = "vf_health"
SCHEMA = "ghana"
VS_ENDPOINT_NAME = "vf_health_vs_endpoint"
VS_INDEX_NAME = f"{CATALOG}.{SCHEMA}.facility_rag_index"

print("\n" + "="*70)
print("🧪 TESTING VECTOR SEARCH INDEX")
print("="*70)
print(f"Endpoint: {VS_ENDPOINT_NAME}")
print(f"Index: {VS_INDEX_NAME}")

vs_client = VectorSearchClient(disable_notice=True)

# Get index with explicit parameters
print("\n🔍 Retrieving index...")
try:
    index = vs_client.get_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=VS_INDEX_NAME
    )
    print("✅ Index retrieved successfully")
    
    # Wait for index to be ready
    print("\n⏳ Checking index status...")
    for i in range(30):
        status = index.describe()
        status_msg = status.get('status', {}).get('message', '')
        
        if 'ready' in status_msg.lower() or 'online' in status_msg.lower():
            print(f"✅ Index is ready: {status_msg}")
            break
        
        print(f"   Waiting for index to be ready... ({i+1}/30) - {status_msg}")
        time.sleep(10)
        
except Exception as e:
    print(f"❌ Error retrieving index: {e}")
    raise

# Test queries focused on medical capabilities
test_queries = [
    "emergency trauma surgery capability",
    "facilities with ICU and intensive care",
    "hospitals in Greater Accra region",
    "medical imaging equipment MRI CT scan",
    "maternal health obstetrics maternity services"
]

print("\n" + "="*70)
print("🔍 RUNNING TEST QUERIES")
print("="*70 + "\n")

for i, query in enumerate(test_queries, 1):
    print(f"{i}. Query: '{query}'")
    print("-" * 70)
    
    try:
        results = index.similarity_search(
            query_text=query,
            columns=["name", "address_region_clean", "facilityTypeId", 
                     "has_emergency", "has_surgery", "has_imaging", 
                     "capability_score"],
            num_results=5
        )
        
        if results and 'result' in results and 'data_array' in results['result']:
            data = results['result']['data_array']
            
            if len(data) > 0:
                print(f"   ✅ Found {len(data)} results:\n")
                
                for j, row in enumerate(data[:3], 1):  # Show top 3
                    print(f"   {j}. {row[0]}")
                    print(f"      Region: {row[1]} | Type: {row[2]}")
                    print(f"      Emergency: {row[3]} | Surgery: {row[4]} | Imaging: {row[5]}")
                    print(f"      Capability Score: {row[6]}/8")
                    print()
            else:
                print("   ⚠️ No results found\n")
        else:
            print("   ⚠️ Unexpected response format\n")
            
    except Exception as e:
        print(f"   ❌ Error: {e}\n")
    
    print()

print("="*70)
print("✓ TEST QUERIES COMPLETE")
print("\n🌟 The vector search index is ready for the LangChain agent!")
print("   Next step: Run 05_build_agent notebook to create the IDP agent")
print("="*70)